In [1]:
# Pandas library import karta hai (CSV aur tabular data handle karne ke liye)
import pandas as pd

# NumPy import karta hai (numerical operations ke liye)
import numpy as np

# File aur folder operations ke liye os module import karta hai
import os

# Confirm karta hai ki sabhi libraries successfully load ho gayi hain
print("Libraries loaded!")

Libraries loaded!


In [2]:
# layout_features.csv file ka path define karta hai
features_path = r"C:\Projects\Internship_Learning\dataset_final\layout_features.csv"

# CSV file ko DataFrame me load karta hai
df = pd.read_csv(features_path)

# DataFrame ki shape (rows, columns) print karta hai
print("Shape:", df.shape)

# DataFrame ke sabhi column names print karta hai
print("\nColumns:", list(df.columns))

# First row display karta hai taaki data verify ho sake
print("\nFirst row:")
print(df.iloc[0])

Shape: (979, 9)

Columns: ['filename', 'num_blocks', 'avg_block_size', 'block_density', 'lr_balance', 'tb_balance', 'text_like_regions', 'image_like_regions', 'layout_clutter_score']

First row:
filename                old_0001.png
num_blocks                        16
avg_block_size               2039.47
block_density                  65.03
lr_balance                      0.37
tb_balance                       0.5
text_like_regions                  9
image_like_regions                 1
layout_clutter_score           37.02
Name: 0, dtype: object


In [3]:
# Handcrafted features CSV file ka path define karta hai
hc_features_path = r"C:\Projects\Internship_Learning\dataset_final\features.csv"

# Handcrafted features CSV ko DataFrame me load karta hai
df_hc = pd.read_csv(hc_features_path)

# DataFrame ki shape (rows, columns) print karta hai
print("Handcrafted features shape:", df_hc.shape)

# Sabhi handcrafted feature columns ke naam print karta hai
print("Columns:", list(df_hc.columns))

Handcrafted features shape: (979, 18)
Columns: ['filename', 'brightness', 'contrast', 'edge_density', 'dominant_color_count', 'whitespace_ratio', 'sharpness', 'saturation_mean', 'dark_area_ratio', 'light_area_ratio', 'clutter_score', 'word_count', 'text_coverage_ratio', 'avg_text_x', 'avg_text_y', 'label', 'score', 'source']


In [4]:
# Filename ke basis par handcrafted aur layout features ko merge karta hai
df_combined = pd.merge(df_hc, df, on='filename', suffixes=('_hc', '_layout'))

# edge_density aur whitespace_ratio CSV me fraction (0-1) me hain, feedback_rules percent (0-100) expect karta hai
df_combined['edge_density'] = df_combined['edge_density'] * 100
df_combined['whitespace_ratio'] = df_combined['whitespace_ratio'] * 100

# Shape aur percent range check
print("Combined shape:", df_combined.shape)
print(df_combined[['edge_density', 'whitespace_ratio', 'text_like_regions']].describe().loc[['min', '50%', 'max']].round(2))

Combined shape: (979, 26)
     edge_density  whitespace_ratio  text_like_regions
min          0.31              0.00                0.0
50%          3.11             35.21                4.0
max         18.18             98.88               26.0


In [5]:
# Shared module feedback_rules.py import karta hai (app.py bhi isi ko use karta hai, isliye rules same rahte hain)
import sys
import importlib

# Project folder ko Python path me jodta hai taaki feedback_rules mil jaye
sys.path.insert(0, r"C:\Projects\Internship_Learning")
import feedback_rules as fr
importlib.reload(fr)   # file badli ho to dobara load karta hai

# Wrapper function: purane notebook cells (6, 7) isi naam se chalte hain
# verbose=True ho to feedback print bhi karta hai
def generate_feedback(features, prediction, verbose=True):

    # Percentile-based rules se feedback list leta hai (max 5)
    feedback = fr.generate_feedback(features)

    # Sirf verbose mode me print karta hai
    if verbose:
        print("=" * 55)
        print(f"Predicted Class  : {str(prediction).upper()}")
        print("=" * 55)
        print("Design Feedback:")
        for i, point in enumerate(feedback, 1):
            print(f"  {i}. {point}")
        print("=" * 55)

    # Prediction aur feedback dictionary return karta hai
    return {'prediction': prediction, 'feedback': feedback}

print("Feedback function defined (percentile-based rules from feedback_rules.py)!")

Feedback function defined (percentile-based rules from feedback_rules.py)!


In [6]:
# Combined DataFrame ki pehli row select karta hai
sample = df_combined.iloc[0]

# Testing ke liye manually predicted class assign karta hai
prediction = "average"

# Test image ka filename print karta hai
print("Testing on:", sample['filename'])

# Ek blank line print karta hai
print()

# Selected sample par feedback function run karta hai
result = generate_feedback(sample, prediction)

Testing on: old_0001.png

Predicted Class  : AVERAGE
Design Feedback:
  1. The page has a lot of fine detail and edges compared to typical pages, so it may feel busy.


In [7]:
# Labels CSV file ka path define karta hai
labels_path = r"C:\Projects\Internship_Learning\dataset_final\labels.csv"

# Labels CSV ko DataFrame me load karta hai
df_labels = pd.read_csv(labels_path)

# features.csv me 'label' pehle se hai, isliye dobara merge nahi karte (label_x / label_y se bachne ke liye)
df_final = df_combined.copy()

# Safety check: features.csv ka label labels.csv ke label se match karta hai ya nahi
check = df_final[['filename', 'label']].merge(
    df_labels[['filename', 'label']], on='filename', suffixes=('_feat', '_lab'))
print("Label mismatches vs labels.csv:", (check['label_feat'] != check['label_lab']).sum())
print("Final dataframe shape:", df_final.shape)

# Sabhi images ka feedback store karne ke liye empty list banata hai
all_feedback = []

# Har row ke liye feedback generate karta hai (verbose=False, taaki 979 blocks print na hon)
for _, row in df_final.iterrows():
    result = generate_feedback(row, row['label'], verbose=False)
    all_feedback.append({
        'filename': row['filename'],
        'prediction': result['prediction'],
        'feedback_1': result['feedback'][0] if len(result['feedback']) > 0 else '',
        'feedback_2': result['feedback'][1] if len(result['feedback']) > 1 else '',
        'feedback_3': result['feedback'][2] if len(result['feedback']) > 2 else '',
        'feedback_4': result['feedback'][3] if len(result['feedback']) > 3 else '',
        'feedback_5': result['feedback'][4] if len(result['feedback']) > 4 else '',
    })

print("\nDone! Feedback generated for", len(all_feedback), "images.")

# Summary: har feedback line kitni images pe aayi (class-wise percent), taaki dikhe ki rules discriminate karte hain ya nahi
fb = pd.DataFrame(all_feedback)
long = fb.melt(id_vars=['filename', 'prediction'], value_vars=[f'feedback_{i}' for i in range(1, 6)])
long = long[long['value'] != '']
summary = (long.groupby(['value', 'prediction']).size().unstack(fill_value=0))
class_counts = fb['prediction'].value_counts()
summary_pct = (summary / class_counts * 100).round(0).astype(int)
print("\nFeedback frequency (% of images in each class):")
print(summary_pct.to_string())

Label mismatches vs labels.csv: 0
Final dataframe shape: (979, 26)

Done! Feedback generated for 979 images.

Feedback frequency (% of images in each class):
prediction                                                                                     average  good  poor
value                                                                                                             
Colors are muted. Consider an accent color for visual interest.                                     15    16    13
Colors are very vivid. High saturation may feel overwhelming.                                       13    18    12
Contrast is low, so text and elements may be harder to distinguish.                                 12    21    13
Many small text-like regions. Consider grouping content into clearer sections.                      17     9    18
The page has a lot of fine detail and edges compared to typical pages, so it may feel busy.         14    10    24
The page has generous empty space, gi

In [8]:
# Generated feedback ko DataFrame me convert karta hai
df_feedback = pd.DataFrame(all_feedback)

# Feedback CSV file save karne ka path define karta hai
feedback_path = r"C:\Projects\Internship_Learning\dataset_final\feedback_results.csv"

# Feedback DataFrame ko CSV file me save karta hai
# index=False -> Extra index column save nahi karta
df_feedback.to_csv(feedback_path, index=False)

# Confirm karta hai ki feedback CSV successfully save ho gayi hai
print("feedback_results.csv saved!")

# Feedback DataFrame ki shape (rows, columns) print karta hai
print("Shape:", df_feedback.shape)

# Sample output dekhne ke liye first 3 rows print karta hai
print("\nSample:")
print(df_feedback.head(3))

feedback_results.csv saved!
Shape: (979, 7)

Sample:
       filename prediction                                         feedback_1  \
0  old_0001.png       good  The page has a lot of fine detail and edges co...   
1  old_0002.png       good                                                      
2  old_0003.png       good                                                      

  feedback_2 feedback_3 feedback_4 feedback_5  
0                                              
1                                              
2                                              
